# Prompt Caching on Amazon Bedrock

This notebook demonstrates:
- Using prompt caching to reduce latency and cost
- Cache checkpoint placement strategies
- Comparing cached vs uncached performance
- Simplified cache management for Claude models
- Amazon Nova automatic caching

## ⚠️ Cost Warning
- **Cache writes** may cost slightly more than standard input tokens
- **Cache reads** cost less than standard input tokens (savings on repeated use)
- Estimated cost for this lab: **< $0.50** (minimal token usage)
- No persistent resources created — no ongoing charges

In [ ]:
# Install required packages
!pip install boto3 -q

In [4]:
import boto3
import json
import time

# Configuration
REGION = "us-west-2"
# Nova Pro for automatic caching demo, Claude for explicit caching demo
NOVA_MODEL_ID = "us.amazon.nova-pro-v1:0"
CLAUDE_MODEL_ID = "us.anthropic.claude-sonnet-4-20250514-v1:0"

bedrock_runtime = boto3.client("bedrock-runtime", region_name=REGION)
print(f"Region: {REGION}")
print(f"Nova Model: {NOVA_MODEL_ID}")
print(f"Claude Model: {CLAUDE_MODEL_ID}")

Region: us-west-2
Nova Model: us.amazon.nova-pro-v1:0
Claude Model: us.anthropic.claude-sonnet-4-20250514-v1:0


## 1. Prepare a Large Context Document

Prompt caching is most effective when you have a large, static context that gets reused across multiple queries. Let's simulate a document Q&A scenario.

In [5]:
# Simulate a large document context (needs to exceed 1024 tokens for Claude caching)
LARGE_CONTEXT = """
# AWS Well-Architected Framework - Operational Excellence Pillar

## Design Principles
There are five design principles for operational excellence in the cloud:

1. Perform operations as code: In the cloud, you can apply the same engineering discipline
that you use for application code to your entire environment. You can define your entire
workload (applications, infrastructure) as code and update it with code. You can implement
your operations procedures as code and automate their execution by triggering them in
response to events.

2. Make frequent, small, reversible changes: Design workloads to allow components to be
updated regularly. Make changes in small increments that can be reversed if they fail
(without affecting customers when possible).

3. Refine operations procedures frequently: As you use operations procedures, look for
opportunities to improve them. As you evolve your workload, evolve your procedures
appropriately. Set up regular game days to review and validate that all procedures are
effective and that teams are familiar with them.

4. Anticipate failure: Perform pre-mortem exercises to identify potential sources of
failure so that they can be removed or mitigated. Test your failure scenarios and validate
your understanding of their impact. Test your response procedures to ensure that they are
effective, and that teams are familiar with their execution.

5. Learn from all operational failures: Drive improvement through lessons learned from all
operational events and failures. Share what is learned across teams and through the entire
organization.

## Best Practices

### Organization
Your teams need to have a shared understanding of your entire workload, their role in it,
and shared business goals to set the priorities that will enable business success.
Well-defined priorities will maximize the benefits of your efforts. Evaluate the needs of
internal and external customers and key stakeholders to determine where to focus efforts.

### Prepare
To prepare for operational excellence, you have to understand your workloads and their
expected behaviors. You will then be able to design them to provide insight to their
status and build the procedures to support them. Design your workload so that it provides
the information necessary for you to understand its internal state across all components.

### Operate
Successful operation of a workload is measured by the achievement of business and customer
outcomes. Define expected outcomes, determine how success will be measured, and identify
the workload and operations metrics that will be used in those calculations to determine
if operations are successful.

### Evolve
You must learn, share, and continuously improve to sustain operational excellence. Dedicate
work cycles to making continuous incremental improvements. Perform post-incident analysis
of all customer-impacting events. Identify contributing factors and preventive action to
limit or prevent recurrence. Communicate contributing factors and corrective actions as
appropriate, targeted to the relevant audience.

## Key AWS Services
The following AWS services and features support the Operational Excellence pillar:
- AWS CloudFormation for infrastructure as code
- AWS Config for configuration compliance
- Amazon CloudWatch for monitoring and observability
- AWS Systems Manager for operations management
- AWS X-Ray for distributed tracing
- Amazon EventBridge for event-driven automation
- AWS CloudTrail for API activity logging
"""

print(f"Context length: ~{len(LARGE_CONTEXT.split())} words")
print(f"Context characters: {len(LARGE_CONTEXT)}")

Context length: ~513 words
Context characters: 3500


## 2. Baseline — Query WITHOUT Caching

First, let's send a query without caching to establish a baseline for latency.

In [6]:
# Baseline: No caching
start_time = time.time()

response_no_cache = bedrock_runtime.converse(
    modelId=CLAUDE_MODEL_ID,
    messages=[
        {
            "role": "user",
            "content": [
                {"text": LARGE_CONTEXT},
                {"text": "What are the 5 design principles for operational excellence?"}
            ]
        }
    ],
    inferenceConfig={"maxTokens": 512, "temperature": 0.1}
)

baseline_latency = time.time() - start_time
baseline_usage = response_no_cache["usage"]

print(f"=== Baseline (No Cache) ===")
print(f"Latency: {baseline_latency:.2f}s")
print(f"Input tokens: {baseline_usage['inputTokens']}")
print(f"Output tokens: {baseline_usage['outputTokens']}")
print(f"\nResponse: {response_no_cache['output']['message']['content'][0]['text'][:200]}...")

=== Baseline (No Cache) ===
Latency: 3.25s
Input tokens: 719
Output tokens: 278

Response: Based on the AWS Well-Architected Framework document, the 5 design principles for operational excellence are:

## 1. Perform operations as code
Apply the same engineering discipline used for applicati...


## 3. Query WITH Cache Checkpoint (First Call = Cache Write)

Now let's add a cache checkpoint after the static context. The first call writes to cache.

In [7]:
# First cached call: This WRITES to cache
start_time = time.time()

response_cache_write = bedrock_runtime.converse(
    modelId=CLAUDE_MODEL_ID,
    messages=[
        {
            "role": "user",
            "content": [
                {"text": LARGE_CONTEXT},
                {"cachePoint": {"type": "default"}},  # <-- Cache everything above this point
                {"text": "What are the 5 design principles for operational excellence?"}
            ]
        }
    ],
    inferenceConfig={"maxTokens": 512, "temperature": 0.1}
)

cache_write_latency = time.time() - start_time
cache_write_usage = response_cache_write["usage"]

print(f"=== Cache Write (First Call) ===")
print(f"Latency: {cache_write_latency:.2f}s")
print(f"Input tokens: {cache_write_usage['inputTokens']}")
print(f"Output tokens: {cache_write_usage['outputTokens']}")
# Cache metrics appear in the usage field
if 'cacheReadInputTokens' in cache_write_usage:
    print(f"Cache read tokens: {cache_write_usage['cacheReadInputTokens']}")
if 'cacheWriteInputTokens' in cache_write_usage:
    print(f"Cache write tokens: {cache_write_usage['cacheWriteInputTokens']}")

=== Cache Write (First Call) ===
Latency: 3.28s
Input tokens: 719
Output tokens: 278
Cache read tokens: 0
Cache write tokens: 0


## 4. Query WITH Cache Hit (Second Call = Cache Read)

Now send a DIFFERENT question with the same context prefix. This should hit the cache — reduced latency and cheaper input tokens.

In [8]:
# Second cached call: This READS from cache (same prefix, different question)
start_time = time.time()

response_cache_read = bedrock_runtime.converse(
    modelId=CLAUDE_MODEL_ID,
    messages=[
        {
            "role": "user",
            "content": [
                {"text": LARGE_CONTEXT},
                {"cachePoint": {"type": "default"}},  # Same cache checkpoint
                {"text": "Which AWS services support the Operational Excellence pillar?"}  # Different question
            ]
        }
    ],
    inferenceConfig={"maxTokens": 512, "temperature": 0.1}
)

cache_read_latency = time.time() - start_time
cache_read_usage = response_cache_read["usage"]

print(f"=== Cache Read (Second Call) ===")
print(f"Latency: {cache_read_latency:.2f}s")
print(f"Input tokens: {cache_read_usage['inputTokens']}")
print(f"Output tokens: {cache_read_usage['outputTokens']}")
if 'cacheReadInputTokens' in cache_read_usage:
    print(f"Cache read tokens: {cache_read_usage['cacheReadInputTokens']}")
if 'cacheWriteInputTokens' in cache_read_usage:
    print(f"Cache write tokens: {cache_read_usage['cacheWriteInputTokens']}")
print(f"\nResponse: {response_cache_read['output']['message']['content'][0]['text'][:200]}...")

=== Cache Read (Second Call) ===
Latency: 2.83s
Input tokens: 718
Output tokens: 241
Cache read tokens: 0
Cache write tokens: 0

Response: Based on the document, the following AWS services support the Operational Excellence pillar:

## Key AWS Services for Operational Excellence

1. **AWS CloudFormation** - for infrastructure as code
2. ...


## 5. Performance Comparison

Let's compare the three approaches side by side.

In [9]:
print("=" * 60)
print(f"{'Metric':<25} {'No Cache':<15} {'Cache Write':<15} {'Cache Read':<15}")
print("=" * 60)
print(f"{'Latency (s)':<25} {baseline_latency:<15.2f} {cache_write_latency:<15.2f} {cache_read_latency:<15.2f}")
print(f"{'Input Tokens':<25} {baseline_usage['inputTokens']:<15} {cache_write_usage['inputTokens']:<15} {cache_read_usage['inputTokens']:<15}")

if cache_read_latency < baseline_latency:
    improvement = ((baseline_latency - cache_read_latency) / baseline_latency) * 100
    print(f"\n✅ Cache read was {improvement:.1f}% faster than uncached baseline")
else:
    print(f"\n⚠️ Cache may not have hit — ensure context exceeds minimum token threshold (1024 for Claude)")

Metric                    No Cache        Cache Write     Cache Read     
Latency (s)               3.25            3.28            2.83           
Input Tokens              719             719             718            

✅ Cache read was 12.9% faster than uncached baseline


## 6. Extended TTL (1-Hour Cache)

For long-running agent sessions or batch processing, use the 1-hour TTL option (available for Claude Sonnet 4.5, Haiku 4.5, Opus 4.5).

In [12]:
# 1-hour TTL example (requires Claude 4.5 models)
# Note: This uses a different billing rate than the 5-minute default

CLAUDE_45_MODEL = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"

try:
    response_1h_ttl = bedrock_runtime.converse(
        modelId=CLAUDE_45_MODEL,
        messages=[
            {
                "role": "user",
                "content": [
                    {"text": LARGE_CONTEXT},
                    {"cachePoint": {"type": "default", "ttl": "1h"}},  # 1-hour TTL
                    {"text": "Summarize the Prepare best practice area."}
                ]
            }
        ],
        inferenceConfig={"maxTokens": 256, "temperature": 0.1}
    )
    print("✅ 1-hour TTL cache write successful")
    print(f"Usage: {response_1h_ttl['usage']}")
except Exception as e:
    print(f"⚠️ 1-hour TTL requires Claude 4.5 models. Error: {e}")

✅ 1-hour TTL cache write successful
Usage: {'inputTokens': 717, 'outputTokens': 135, 'totalTokens': 852, 'cacheReadInputTokens': 0, 'cacheWriteInputTokens': 0}


## 7. System Prompt Caching

You can also cache system prompts — useful when the same system prompt is reused across many conversations.

In [13]:
# Cache a large system prompt
SYSTEM_PROMPT = f"""You are an AWS Solutions Architect assistant. Use the following reference 
material to answer questions accurately and concisely.

Reference Material:
{LARGE_CONTEXT}

Instructions:
- Always cite specific sections from the reference material
- If the answer is not in the reference material, say so
- Keep answers concise and actionable
"""

start_time = time.time()

response_system_cache = bedrock_runtime.converse(
    modelId=CLAUDE_MODEL_ID,
    system=[
        {"text": SYSTEM_PROMPT},
        {"cachePoint": {"type": "default"}}  # Cache the system prompt
    ],
    messages=[
        {
            "role": "user",
            "content": [{"text": "What does 'Perform operations as code' mean?"}]
        }
    ],
    inferenceConfig={"maxTokens": 256, "temperature": 0.1}
)

system_cache_latency = time.time() - start_time
print(f"System prompt cached. Latency: {system_cache_latency:.2f}s")
print(f"Usage: {response_system_cache['usage']}")
print(f"\nResponse: {response_system_cache['output']['message']['content'][0]['text'][:300]}")

System prompt cached. Latency: 3.29s
Usage: {'inputTokens': 782, 'outputTokens': 239, 'totalTokens': 1021, 'cacheReadInputTokens': 0, 'cacheWriteInputTokens': 0}

Response: According to the AWS Well-Architected Framework's Operational Excellence pillar, "Perform operations as code" means:

**In the cloud, you can apply the same engineering discipline that you use for application code to your entire environment. You can define your entire workload (applications, infrast


## 8. Amazon Nova — Automatic Caching

Nova models provide automatic prompt caching with no explicit configuration needed. Latency benefits occur when prompts begin with repetitive parts.

In [17]:
# Nova automatic caching — no cache checkpoints needed
# Send the same context twice and observe latency improvement

# First call
start_time = time.time()
response_nova_1 = bedrock_runtime.converse(
    modelId=NOVA_MODEL_ID,
    messages=[
        {
            "role": "user",
            "content": [
                {"text": LARGE_CONTEXT},
                {"text": "List the 5 design principles."}
            ]
        }
    ],
    inferenceConfig={"maxTokens": 256, "temperature": 0.1}
)
nova_first_latency = time.time() - start_time

# Second call (same prefix — Nova may auto-cache)
start_time = time.time()
response_nova_2 = bedrock_runtime.converse(
    modelId=NOVA_MODEL_ID,
    messages=[
        {
            "role": "user",
            "content": [
                {"text": LARGE_CONTEXT},
                {"text": "What are the key AWS services mentioned?"}
            ]
        }
    ],
    inferenceConfig={"maxTokens": 256, "temperature": 0.1}
)
nova_second_latency = time.time() - start_time

print(f"=== Nova Automatic Caching ===")
print(f"First call latency:  {nova_first_latency:.2f}s")
print(f"Second call latency: {nova_second_latency:.2f}s")
if nova_second_latency < nova_first_latency:
    print(f"✅ {((nova_first_latency - nova_second_latency) / nova_first_latency * 100):.1f}% latency improvement (auto-cache likely hit)")
else:
    print("ℹ️ No improvement observed — auto-caching benefits vary by workload")

=== Nova Automatic Caching ===
First call latency:  1.55s
Second call latency: 1.87s
ℹ️ No improvement observed — auto-caching benefits vary by workload


## 9. Best Practices Summary

| Strategy | When to Use |
|----------|-------------|
| **Explicit cache checkpoints** | Large static context reused across queries (Claude) |
| **System prompt caching** | Same system prompt across many conversations |
| **1-hour TTL** | Long-running agents, batch processing, infrequent interactions |
| **Nova auto-caching** | Any Nova workload with repetitive prefixes (free) |
| **Simplified management** | Single checkpoint at end of static content (Claude auto-finds best match) |

### Key Rules
- Cache prefix must be **static** between requests (any change = cache miss)
- Minimum **1,024 tokens** per checkpoint (Claude)
- Maximum **4 checkpoints** per request (Claude)
- Default TTL: **5 minutes** (resets on each hit)
- Place dynamic content **after** the cache checkpoint

## 🧹 Cleanup (Optional)

No persistent resources were created in this lab. Cached content expires automatically after the TTL window (5 minutes or 1 hour). No cleanup needed.